# 🩺 Notebook 1 — Diabetes Dataset EDA
**Dataset**: Pima Indians Diabetes Database  
**Source**: [Kaggle — UCI ML](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)  
**Rows**: 768 | **Features**: 8 | **Target**: `Outcome` (0 = No Diabetes, 1 = Diabetes)

### Objective
Perform full Exploratory Data Analysis on the diabetes dataset to understand:
- Class imbalance
- Biological impossibilities (zero values)
- Feature distributions by outcome
- Correlations between features
- Key signals for the cascade model


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# ── Colour palette ──────────────────────────────────────────────
BLUE   = "#2D6A9F"
RED    = "#C0392B"
GREEN  = "#1A7A4A"
GREY   = "#85929E"
BG     = "#F7F9FB"
DARK   = "#1C2833"

np.random.seed(42)
print("Libraries loaded ✅")


## 1 · Load & Synthesise Dataset
> **Note**: Since we cannot download from Kaggle directly here, we synthesise a statistically faithful replica (same schema, similar distributions, same zero-injection pattern as the real Pima dataset). Replace `df = make_diabetes()` with `df = pd.read_csv('diabetes.csv')` when running with the real file.


In [ ]:
def make_diabetes():
    """Synthesise Pima Indians Diabetes data (768 rows)."""
    n = 768
    neg, pos = 500, 268          # ~65.1% negative, ~34.9% positive

    def col(neg_mu, neg_sd, pos_mu, pos_sd, low=None):
        v = np.concatenate([
            np.random.normal(neg_mu, neg_sd, neg),
            np.random.normal(pos_mu, pos_sd, pos)
        ])
        if low is not None:
            v = np.clip(v, low, None)
        return v

    df = pd.DataFrame({
        'Pregnancies':              col(3.3, 3.0, 4.9, 3.5, 0).astype(int),
        'Glucose':                  col(109, 26, 141, 31, 0),
        'BloodPressure':            col(68, 18, 75, 20, 0),
        'SkinThickness':            col(20, 15, 32, 17, 0),
        'Insulin':                  col(68, 98, 100, 140, 0),
        'BMI':                      col(30.9, 7.5, 35.1, 7.4, 0),
        'DiabetesPedigreeFunction': col(0.43, 0.29, 0.55, 0.37, 0.078),
        'Age':                      col(31, 11, 37, 11, 21).astype(int),
        'Outcome':                  np.array([0]*neg + [1]*pos)
    })

    # Inject biologically impossible zeros (mirrors real Pima dataset)
    for col_name, frac in [('Glucose', 0.007), ('BloodPressure', 0.046),
                            ('BMI', 0.014), ('Insulin', 0.49), ('SkinThickness', 0.30)]:
        idx = df.sample(frac=frac).index
        df.loc[idx, col_name] = 0

    return df.sample(frac=1, random_state=42).reset_index(drop=True)

df = make_diabetes()
# df = pd.read_csv('diabetes.csv')   # ← use this line with the real file

print(f"Shape: {df.shape}")
df.head()


## 2 · Basic Statistics

In [ ]:
print("=== DTYPES ===")
print(df.dtypes)
print("\n=== DESCRIBE ===")
df.describe().round(2)


In [ ]:
print("=== MISSING VALUES (actual NaN) ===")
print(df.isnull().sum())
print("\n=== BIOLOGICAL ZEROS (treated as missing) ===")
bio_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for c in bio_cols:
    z = (df[c] == 0).sum()
    pct = z / len(df) * 100
    print(f"  {c:30s}: {z:3d} zeros  ({pct:.1f}%)")


## 3 · Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor=BG)

# Bar chart
counts = df['Outcome'].value_counts().sort_index()
labels = ['No Diabetes', 'Diabetes']
bars = axes[0].bar(labels, counts.values, color=[GREEN, RED],
                    edgecolor='white', linewidth=1.2, width=0.5)
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+3, str(v),
                 ha='center', va='bottom', fontsize=12, fontweight='bold', color=DARK)
axes[0].set_title('Class Distribution — Count', fontweight='bold', color=DARK)
axes[0].set_ylabel('Count'); axes[0].set_facecolor(BG)
axes[0].spines[['top','right']].set_visible(False)

# Pie chart
axes[1].pie(counts.values, labels=labels, colors=[GREEN, RED],
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Distribution — Proportion', fontweight='bold', color=DARK)
axes[1].set_facecolor(BG)

plt.tight_layout(); plt.show()
print(f"Class balance: {counts[0]} negative vs {counts[1]} positive")
print(f"Imbalance ratio: {counts[0]/counts[1]:.2f}:1")


## 4 · Feature Distributions by Outcome

In [ ]:
features = ['Glucose', 'BloodPressure', 'SkinThickness',
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Pregnancies']

fig, axes = plt.subplots(2, 4, figsize=(18, 8), facecolor=BG)
axes = axes.flatten()

col_map = {0: GREEN, 1: RED}
label_map = {0: 'No Diabetes', 1: 'Diabetes'}

for ax, feat in zip(axes, features):
    for outcome, clr in col_map.items():
        vals = df[df['Outcome'] == outcome][feat]
        ax.hist(vals, bins=25, alpha=0.65, color=clr,
                label=label_map[outcome], edgecolor='white', linewidth=0.4)
    ax.set_title(feat, fontweight='bold', color=DARK, fontsize=10)
    ax.set_xlabel('Value', fontsize=8); ax.set_ylabel('Frequency', fontsize=8)
    ax.legend(fontsize=7); ax.set_facecolor(BG)
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('Feature Distributions: Diabetes vs No Diabetes',
             fontsize=14, fontweight='bold', color=DARK, y=1.01)
plt.tight_layout(); plt.show()


## 5 · Boxplots — Feature Spread by Class

In [ ]:
key_features = ['Glucose', 'BMI', 'Age', 'BloodPressure',
                'Insulin', 'DiabetesPedigreeFunction']

fig, axes = plt.subplots(2, 3, figsize=(16, 8), facecolor=BG)
axes = axes.flatten()

for ax, feat in zip(axes, key_features):
    data_neg = df[df['Outcome']==0][feat]
    data_pos = df[df['Outcome']==1][feat]
    bp = ax.boxplot([data_neg, data_pos], patch_artist=True, notch=True,
                    medianprops=dict(color='white', linewidth=2.5))
    for patch, clr in zip(bp['boxes'], [GREEN, RED]):
        patch.set_facecolor(clr); patch.set_alpha(0.75)
    ax.set_xticklabels(['No Diabetes', 'Diabetes'], fontsize=9)
    ax.set_title(feat, fontweight='bold', color=DARK)
    ax.set_facecolor(BG); ax.spines[['top','right']].set_visible(False)

fig.suptitle('Boxplots by Outcome — Key Features',
             fontsize=14, fontweight='bold', color=DARK)
plt.tight_layout(); plt.show()


## 6 · Correlation Matrix

In [ ]:
import matplotlib.patches as mpatches

corr = df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8), facecolor=BG)
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns, fontsize=9)

for i in range(len(corr)):
    for j in range(len(corr)):
        val = corr.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(val) > 0.4 else DARK, fontweight='bold')

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Correlation Matrix — Diabetes Dataset',
             fontweight='bold', color=DARK, fontsize=13, pad=12)
plt.tight_layout(); plt.show()

# Top correlations with Outcome
print("\nTop correlations with Outcome:")
print(corr['Outcome'].sort_values(ascending=False).to_string())


## 7 · Scatter Plots — Key Feature Pairs

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)
pairs = [('Glucose', 'BMI'), ('Age', 'Glucose'), ('BMI', 'Insulin')]

for ax, (fx, fy) in zip(axes, pairs):
    for outcome, clr in col_map.items():
        sub = df[df['Outcome'] == outcome]
        ax.scatter(sub[fx], sub[fy], c=clr, alpha=0.45, s=20,
                   label=label_map[outcome], edgecolors='none')
    ax.set_xlabel(fx, fontsize=10); ax.set_ylabel(fy, fontsize=10)
    ax.set_title(f'{fx} vs {fy}', fontweight='bold', color=DARK)
    ax.legend(fontsize=8); ax.set_facecolor(BG)
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('Scatter Plots — Key Feature Pairs', fontsize=14, fontweight='bold', color=DARK)
plt.tight_layout(); plt.show()


## 8 · Zero-Value Analysis (Biological Impossibilities)
Zero values in `Glucose`, `BloodPressure`, `BMI`, `Insulin`, `SkinThickness` are biologically impossible.  
These will be treated as **NaN** and imputed using **median imputation per outcome class** in Phase 1 of the pipeline.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor=BG)
cols_to_check = ['Glucose', 'Insulin', 'BMI']

for ax, col_name in zip(axes, cols_to_check):
    non_zero = df[df[col_name] > 0][col_name]
    zero_count = (df[col_name] == 0).sum()
    ax.hist(non_zero, bins=30, color=BLUE, alpha=0.75, edgecolor='white', linewidth=0.4)
    ax.axvline(non_zero.median(), color=RED, linestyle='--', linewidth=2, label=f'Median: {non_zero.median():.1f}')
    ax.set_title(f'{col_name}\n({zero_count} zeros removed)', fontweight='bold', color=DARK)
    ax.set_xlabel('Value'); ax.set_ylabel('Frequency')
    ax.legend(fontsize=8); ax.set_facecolor(BG)
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('Distributions After Excluding Zero Values\n(Zeros = Biologically Impossible → Will Be Imputed)',
             fontsize=13, fontweight='bold', color=DARK)
plt.tight_layout(); plt.show()


## 9 · Summary for Phase 1 Pipeline

| Feature | Missing (zeros) | Imputation Strategy |
|---|---|---|
| Glucose | ~0.7% | Median per class |
| BloodPressure | ~4.6% | Median per class |
| BMI | ~1.4% | Median per class |
| Insulin | ~49% | Median per class (high miss rate — consider model-based) |
| SkinThickness | ~30% | Median per class |

**Key signals for cascade model:**
- `Glucose` → strongest predictor of diabetes, will be forwarded into hypertension model
- `BMI` → shared amplifier, forwarded into all downstream models
- `DiabetesPedigreeFunction` → genetic marker (fixed feature, not modifiable)
- `Age` → universal confounder
